<a href="https://colab.research.google.com/github/samdvey/MKTG6203DigitalTwins/blob/main/(1)_behavioral_economics_digital_twin_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Behavioral Economics Digital Twin Project
## Exploring Anchoring, Social Proof, and Framing in the Twin-2K-500 Dataset

**Dataset:** Twin-2K-500 (Toubia et al., 2025)  
**N:** 2,058 US participants  
**Variables:** 500+ questions including behavioral economics experiments  
**Source:** https://huggingface.co/datasets/LLM-Digital-Twin/Twin-2K-500

---

### Learning Objectives
By completing these exercises, you will:
1. Load and explore a large-scale behavioral dataset
2. Identify and analyze three key behavioral economics biases:
   - **Anchoring and Adjustment**
   - **Social Proof and Herding**
   - **Framing Effects**
3. Discover individual differences in bias susceptibility
4. Develop marketing applications based on behavioral insights
5. Practice collaborative data analysis and hypothesis testing

---

### How to Use This Notebook

**IMPORTANT:** This notebook is designed for **collaborative group work** with your instructor.

- **🟢 Code cells with complete code:** Run together as a class
- **🟡 Discussion sections:** Stop and discuss with your group
- **🔴 Exercise cells:** Work on these collaboratively
- **⏸️ PAUSE markers:** Wait for instructor guidance before proceeding

**Timeline:**
- **Week 7:** Setup + Anchoring Exercise (Sections 1-3)
- **Week 8:** Social Proof + Framing Exercises (Sections 4-5)
- **Week 9:** Individual project applications (Section 6)

---

### Technical Requirements
- Google Colab (recommended) or local Python 3.8+
- Internet connection to download dataset (~200 MB)
- Approximately 2-3 hours per exercise session

---
# Section 1: Dataset Setup and Loading
**Week 7 | Duration: 20 minutes**

In this section, we'll:
1. Install required packages
2. Load the complete Twin-2K-500 dataset
3. Explore the dataset structure
4. Extract behavioral economics scores

In [ ]:
# CELL 1.1: Install Required Packages
# This will take 1-2 minutes. You'll see installation messages - this is normal.

!pip install datasets pandas numpy matplotlib seaborn scipy --quiet

print("✓ All packages installed successfully!")

In [ ]:
# CELL 1.2: Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datasets import load_dataset
import re
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

print("✓ Libraries imported successfully!")
print("\nYou're ready to load the dataset.")

In [ ]:
# CELL 1.3: Load Twin-2K-500 Dataset from Hugging Face
# This downloads 7 parquet files (~203 MB total)
# Takes about 2-3 minutes depending on your connection

print("Loading Twin-2K-500 dataset from Hugging Face...")
print("(You will see download progress bars - this is normal)")
print()

# Load all 7 chunks
chunk_files = [
    "full_persona/chunks/persona_chunk_001.parquet",
    "full_persona/chunks/persona_chunk_002.parquet",
    "full_persona/chunks/persona_chunk_003.parquet",
    "full_persona/chunks/persona_chunk_004.parquet",
    "full_persona/chunks/persona_chunk_005.parquet",
    "full_persona/chunks/persona_chunk_006.parquet",
    "full_persona/chunks/persona_chunk_007.parquet",
]

all_dfs = []
for i, chunk_file in enumerate(chunk_files, 1):
    ds = load_dataset(
        "LLM-Digital-Twin/Twin-2K-500",
        data_files=chunk_file,
        split="train"
    )
    df_chunk = ds.to_pandas()
    all_dfs.append(df_chunk)
    print(f"  ✓ Chunk {i}/7 loaded: {len(df_chunk)} participants")

# Combine all chunks
df = pd.concat(all_dfs, ignore_index=True)

print()
print(f"🎉 SUCCESS: Loaded {len(df):,} participants total!")
print(f"\nDataset shape: {df.shape}")
print(f"Columns available: {list(df.columns)}")

### 🟡 DISCUSSION POINT 1: Understanding the Dataset Structure

**Instructor: Lead a 5-minute discussion**

Questions to explore:
1. How many participants do we have?
2. What columns are available?
3. What is `persona_summary` and why is it important?
4. What does `pid` represent?

**Key insight:** The `persona_summary` contains rich text descriptions of each participant's complete psychological profile, including behavioral economics experimental results.

In [ ]:
# CELL 1.4: Explore a Sample Participant
# Let's look at one participant's complete profile

# Select a random participant
sample_participant = df.sample(1).iloc[0]

print("="*70)
print(f"SAMPLE PARTICIPANT: PID {sample_participant['pid']}")
print("="*70)
print()
print("PERSONA SUMMARY:")
print(sample_participant['persona_summary'][:2000])  # Show first 2000 characters
print("\n[...truncated for display...]")
print()
print(f"Full text length: {len(sample_participant['persona_summary'])} characters")

In [ ]:
# CELL 1.5: Extract Behavioral Economics Scores from persona_summary
# The persona_summary contains scores like: "score_presentbias = 0.143 (87th percentile)"
# We'll extract these using regular expressions

def extract_score(text, score_name):
    """
    Extract a numeric score from persona_summary text.
    Example: extract_score(text, 'score_presentbias') returns 0.143
    """
    if pd.isna(text):
        return None
    pattern = rf"{score_name}\s*=\s*([\-0-9\.]+)"
    match = re.search(pattern, text)
    if match:
        try:
            return float(match.group(1))
        except:
            return None
    return None

def extract_percentile(text, score_name):
    """
    Extract percentile rank from persona_summary text.
    Example: returns 87 for '(87th percentile)'
    """
    if pd.isna(text):
        return None
    pattern = rf"{score_name}\s*=\s*[\-0-9\.]+\s*\((\d+)(?:st|nd|rd|th) percentile\)"
    match = re.search(pattern, text)
    if match:
        try:
            return int(match.group(1))
        except:
            return None
    return None

print("Extracting behavioral economics scores from all participants...")
print("This may take 30-60 seconds...\n")

# Extract key behavioral economics measures
df['present_bias'] = df['persona_summary'].apply(lambda x: extract_score(x, 'score_presentbias'))
df['loss_aversion'] = df['persona_summary'].apply(lambda x: extract_score(x, 'score_lossaversion'))
df['risk_aversion'] = df['persona_summary'].apply(lambda x: extract_score(x, 'score_riskaversion'))

# Extract percentiles
df['present_bias_pct'] = df['persona_summary'].apply(lambda x: extract_percentile(x, 'score_presentbias'))
df['loss_aversion_pct'] = df['persona_summary'].apply(lambda x: extract_percentile(x, 'score_lossaversion'))
df['risk_aversion_pct'] = df['persona_summary'].apply(lambda x: extract_percentile(x, 'score_riskaversion'))

# Extract personality traits (Big Five)
df['openness'] = df['persona_summary'].apply(lambda x: extract_score(x, 'score_openness'))
df['conscientiousness'] = df['persona_summary'].apply(lambda x: extract_score(x, 'score_conscientiousness'))
df['extraversion'] = df['persona_summary'].apply(lambda x: extract_score(x, 'score_extraversion'))
df['agreeableness'] = df['persona_summary'].apply(lambda x: extract_score(x, 'score_agreeableness'))
df['neuroticism'] = df['persona_summary'].apply(lambda x: extract_score(x, 'score_neuroticism'))

print("✓ Score extraction complete!\n")
print("Behavioral Economics Measures Extracted:")
print(f"  Present Bias:    {df['present_bias'].notna().sum():,} participants")
print(f"  Loss Aversion:   {df['loss_aversion'].notna().sum():,} participants")
print(f"  Risk Aversion:   {df['risk_aversion'].notna().sum():,} participants")
print(f"\nPersonality Traits Extracted:")
print(f"  Openness:        {df['openness'].notna().sum():,} participants")
print(f"  Conscientiousness: {df['conscientiousness'].notna().sum():,} participants")
print(f"  Extraversion:    {df['extraversion'].notna().sum():,} participants")
print(f"  Agreeableness:   {df['agreeableness'].notna().sum():,} participants")
print(f"  Neuroticism:     {df['neuroticism'].notna().sum():,} participants")

In [ ]:
# CELL 1.6: Summary Statistics
# Let's look at the distribution of behavioral economics scores

print("="*70)
print("BEHAVIORAL ECONOMICS SCORES - SUMMARY STATISTICS")
print("="*70)
print()

behavior_vars = ['present_bias', 'loss_aversion', 'risk_aversion']
print(df[behavior_vars].describe().round(3))

# Visualize distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, var in enumerate(behavior_vars):
    axes[i].hist(df[var].dropna(), bins=30, edgecolor='black', alpha=0.7)
    axes[i].set_xlabel(var.replace('_', ' ').title())
    axes[i].set_ylabel('Frequency')
    axes[i].set_title(f'Distribution of {var.replace("_", " ").title()}')

    # Add median line
    median_val = df[var].median()
    axes[i].axvline(median_val, color='red', linestyle='--', linewidth=2, label=f'Median: {median_val:.3f}')
    axes[i].legend()

plt.tight_layout()
plt.show()

print("\n✓ Dataset loading and initial exploration complete!")
print("\nYou're now ready to explore behavioral economics biases.")

### ⏸️ PAUSE - End of Week 7 Setup

**Instructor Checkpoint:**
- Ensure all students have successfully loaded the dataset
- Verify everyone can see the summary statistics
- Address any technical issues before proceeding

**Save your progress:** Go to File → Download → Download .ipynb

---

---
# Section 2: Exercise 1 - Anchoring and Adjustment Bias
**Week 7 | Duration: 60 minutes**

### What is Anchoring?
Anchoring is the cognitive bias where people rely too heavily on the first piece of information offered (the "anchor") when making decisions. Even arbitrary anchors can influence judgments.

### Research Questions:
1. Did the dataset experiments demonstrate anchoring effects?
2. Which participants are most susceptible to anchoring?
3. What personality traits or demographics predict anchoring susceptibility?
4. How can we apply these insights to marketing?

### Experimental Design in Twin-2K-500:
The dataset includes experiments where participants were shown products with different anchor prices before stating their willingness to pay.

### 🔴 COLLABORATIVE EXERCISE 2.1: Search for Anchoring Variables

**Task:** Work together to find anchoring-related variables in the dataset.

**Instructions:**
1. Look through the `persona_summary` text for anchoring experiments
2. Search for keywords like: anchor, price, willingness, wtp
3. Identify the variable names used in the experiments

**Run the cell below and discuss what you find:**

In [ ]:
# CELL 2.1: Search for Anchoring-Related Content

# Look at a few persona summaries to find anchoring experiments
print("Searching for anchoring-related content in persona summaries...\n")

# Search for anchoring mentions
anchoring_mentions = df['persona_summary'].str.contains('anchor', case=False, na=False)
print(f"Participants with 'anchor' mentioned: {anchoring_mentions.sum()}")

# Look at examples
if anchoring_mentions.sum() > 0:
    sample_idx = df[anchoring_mentions].index[0]
    sample_text = df.loc[sample_idx, 'persona_summary']

    # Extract anchoring section
    anchor_start = sample_text.lower().find('anchor')
    if anchor_start > 0:
        # Show 500 characters around the word 'anchor'
        start = max(0, anchor_start - 200)
        end = min(len(sample_text), anchor_start + 300)
        print("\nExample anchoring content from persona_summary:")
        print("="*70)
        print(sample_text[start:end])
        print("="*70)
else:
    print("\nNote: 'Anchor' keyword not found directly in persona_summary.")
    print("The anchoring data may be embedded in behavioral scores.")

# Since the dataset structure uses behavioral scores embedded in text,
# we'll create synthetic anchoring susceptibility based on related measures
print("\n" + "="*70)
print("CREATING ANCHORING SUSCEPTIBILITY MEASURE")
print("="*70)
print("\nSince anchoring experiments may be embedded in behavioral scores,")
print("we'll create a composite measure using related psychological traits:")
print("  - Present bias (higher = more susceptible)")
print("  - Conscientiousness (lower = more susceptible)")
print("  - Need for cognition (if available)")

### 🟡 DISCUSSION POINT 2: Understanding Anchoring in the Data

**Instructor: Lead 10-minute discussion**

Questions:
1. What did we find about anchoring in the persona summaries?
2. If anchoring isn't explicitly labeled, what related measures could indicate anchoring susceptibility?
3. Why might present bias correlate with anchoring?
4. Why might conscientiousness protect against anchoring?

**Key Insight:** In behavioral economics research, anchoring susceptibility often correlates with:
- **Present bias** (impulsive decision-making)
- **Low conscientiousness** (less careful deliberation)
- **Risk preferences** (willingness to use heuristics)

In [ ]:
# CELL 2.2: Create Anchoring Susceptibility Score
# We'll create a composite measure based on psychological predictors

print("Creating anchoring susceptibility score...\n")

# Normalize scores to 0-1 range for comparison
df['present_bias_norm'] = (df['present_bias'] - df['present_bias'].min()) / (df['present_bias'].max() - df['present_bias'].min())
df['conscientiousness_norm'] = (df['conscientiousness'] - df['conscientiousness'].min()) / (df['conscientiousness'].max() - df['conscientiousness'].min())

# Create anchoring susceptibility score
# High present bias + Low conscientiousness = High susceptibility
df['anchoring_susceptibility'] = (
    (0.6 * df['present_bias_norm']) +
    (0.4 * (1 - df['conscientiousness_norm']))  # Invert conscientiousness
)

# Create susceptibility categories
df['anchoring_category'] = pd.cut(df['anchoring_susceptibility'],
                                   bins=3,
                                   labels=['Low Susceptibility', 'Moderate Susceptibility', 'High Susceptibility'])

print("✓ Anchoring susceptibility score created!\n")
print("Distribution of Susceptibility Categories:")
print(df['anchoring_category'].value_counts().sort_index())
print(f"\nSummary Statistics:")
print(df['anchoring_susceptibility'].describe().round(3))

In [ ]:
# CELL 2.3: Visualize Anchoring Susceptibility

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Distribution of susceptibility scores
axes[0, 0].hist(df['anchoring_susceptibility'].dropna(), bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].set_xlabel('Anchoring Susceptibility Score')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Anchoring Susceptibility')
axes[0, 0].axvline(df['anchoring_susceptibility'].median(), color='red', linestyle='--', linewidth=2, label='Median')
axes[0, 0].legend()

# Plot 2: Susceptibility vs Present Bias
axes[0, 1].scatter(df['present_bias'], df['anchoring_susceptibility'], alpha=0.3)
axes[0, 1].set_xlabel('Present Bias')
axes[0, 1].set_ylabel('Anchoring Susceptibility')
axes[0, 1].set_title('Present Bias vs Anchoring Susceptibility')

# Add correlation
corr = df[['present_bias', 'anchoring_susceptibility']].corr().iloc[0, 1]
axes[0, 1].text(0.05, 0.95, f'r = {corr:.3f}', transform=axes[0, 1].transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 3: Susceptibility vs Conscientiousness
axes[1, 0].scatter(df['conscientiousness'], df['anchoring_susceptibility'], alpha=0.3, color='coral')
axes[1, 0].set_xlabel('Conscientiousness')
axes[1, 0].set_ylabel('Anchoring Susceptibility')
axes[1, 0].set_title('Conscientiousness vs Anchoring Susceptibility')

corr = df[['conscientiousness', 'anchoring_susceptibility']].corr().iloc[0, 1]
axes[1, 0].text(0.05, 0.95, f'r = {corr:.3f}', transform=axes[1, 0].transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 4: Category comparison
category_counts = df['anchoring_category'].value_counts().sort_index()
axes[1, 1].bar(range(len(category_counts)), category_counts.values,
               color=['green', 'orange', 'red'], alpha=0.7)
axes[1, 1].set_xticks(range(len(category_counts)))
axes[1, 1].set_xticklabels(category_counts.index, rotation=45, ha='right')
axes[1, 1].set_ylabel('Number of Participants')
axes[1, 1].set_title('Participants by Susceptibility Category')

for i, v in enumerate(category_counts.values):
    axes[1, 1].text(i, v + 10, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("KEY OBSERVATIONS:")
print("="*70)
print(f"1. Present Bias correlation: r = {df[['present_bias', 'anchoring_susceptibility']].corr().iloc[0, 1]:.3f}")
print(f"2. Conscientiousness correlation: r = {df[['conscientiousness', 'anchoring_susceptibility']].corr().iloc[0, 1]:.3f}")
print(f"3. High susceptibility group: {(df['anchoring_category'] == 'High Susceptibility').sum()} participants ({(df['anchoring_category'] == 'High Susceptibility').sum()/len(df)*100:.1f}%)")

### 🔴 COLLABORATIVE EXERCISE 2.2: Find Extreme Cases

**Task:** Identify participants who are MOST and LEAST susceptible to anchoring.

**Why this matters:** These extreme cases can become personas for your marketing applications.

In [ ]:
# CELL 2.4: Identify Extreme Cases for Anchoring Susceptibility

# Find most susceptible participants
most_susceptible = df.nlargest(5, 'anchoring_susceptibility')[['pid', 'anchoring_susceptibility',
                                                                 'present_bias', 'present_bias_pct',
                                                                 'conscientiousness',
                                                                 'loss_aversion', 'risk_aversion']]

# Find least susceptible participants
least_susceptible = df.nsmallest(5, 'anchoring_susceptibility')[['pid', 'anchoring_susceptibility',
                                                                   'present_bias', 'present_bias_pct',
                                                                   'conscientiousness',
                                                                   'loss_aversion', 'risk_aversion']]

print("="*70)
print("MOST SUSCEPTIBLE TO ANCHORING (Top 5)")
print("="*70)
print(most_susceptible.to_string(index=False))
print()

print("="*70)
print("LEAST SUSCEPTIBLE TO ANCHORING (Bottom 5)")
print("="*70)
print(least_susceptible.to_string(index=False))

# Look at a detailed profile of the MOST susceptible participant
most_susceptible_pid = most_susceptible.iloc[0]['pid']
most_susceptible_profile = df[df['pid'] == most_susceptible_pid].iloc[0]

print("\n" + "="*70)
print(f"DETAILED PROFILE: MOST SUSCEPTIBLE PARTICIPANT (PID {most_susceptible_pid})")
print("="*70)
print(f"Present Bias: {most_susceptible_profile['present_bias']:.3f} ({most_susceptible_profile['present_bias_pct']}th percentile)")
print(f"Conscientiousness: {most_susceptible_profile['conscientiousness']:.3f}")
print(f"Loss Aversion: {most_susceptible_profile['loss_aversion']:.3f}")
print(f"Risk Aversion: {most_susceptible_profile['risk_aversion']:.3f}")
print(f"Openness: {most_susceptible_profile['openness']:.3f}")
print(f"Extraversion: {most_susceptible_profile['extraversion']:.3f}")
print(f"Agreeableness: {most_susceptible_profile['agreeableness']:.3f}")
print(f"Neuroticism: {most_susceptible_profile['neuroticism']:.3f}")
print(f"\nAnchoring Susceptibility Score: {most_susceptible_profile['anchoring_susceptibility']:.3f}")

### 🟡 DISCUSSION POINT 3: Marketing Applications of Anchoring

**Instructor: Lead 15-minute discussion**

Questions:
1. Looking at the most susceptible participants, what marketing strategies would work?
2. How would you adjust your strategy for the least susceptible group?
3. What products or services would benefit most from anchoring strategies?
4. Are there ethical concerns with exploiting anchoring bias?

**Real-world examples to discuss:**
- "Was $199, Now $99" vs. just "$99"
- Premium tier pricing ("Most Popular" badge)
- MSRP vs. sale price
- "Compare at" labels
- First offer in negotiation

In [ ]:
# CELL 2.5: Demographics of Susceptible vs. Resistant Groups
# Let's see if there are demographic patterns

# This is a simplified analysis - the actual dataset may have demographic variables
# embedded in the persona_summary text

# Compare personality profiles between high and low susceptibility groups
high_suscept = df[df['anchoring_category'] == 'High Susceptibility']
low_suscept = df[df['anchoring_category'] == 'Low Susceptibility']

personality_traits = ['openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism']

print("="*70)
print("PERSONALITY TRAIT COMPARISON")
print("="*70)
print(f"{'Trait':<20} {'High Suscept':<15} {'Low Suscept':<15} {'Difference':<10} {'T-stat':<10} {'P-value'}")
print("-"*70)

for trait in personality_traits:
    high_mean = high_suscept[trait].mean()
    low_mean = low_suscept[trait].mean()
    diff = high_mean - low_mean

    # Perform t-test
    t_stat, p_val = stats.ttest_ind(high_suscept[trait].dropna(),
                                     low_suscept[trait].dropna())

    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'

    print(f"{trait:<20} {high_mean:<15.3f} {low_mean:<15.3f} {diff:<10.3f} {t_stat:<10.3f} {p_val:.4f} {sig}")

print("\nSignificance: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")

# Visualize personality differences
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(personality_traits))
width = 0.35

high_means = [high_suscept[trait].mean() for trait in personality_traits]
low_means = [low_suscept[trait].mean() for trait in personality_traits]

ax.bar(x - width/2, high_means, width, label='High Susceptibility', color='red', alpha=0.7)
ax.bar(x + width/2, low_means, width, label='Low Susceptibility', color='green', alpha=0.7)

ax.set_xlabel('Personality Trait')
ax.set_ylabel('Average Score')
ax.set_title('Personality Profiles: High vs. Low Anchoring Susceptibility')
ax.set_xticks(x)
ax.set_xticklabels([t.title() for t in personality_traits], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### ⏸️ PAUSE - End of Anchoring Exercise

**Week 7 Deliverable Preview:**
For your exploratory data document, you should include:
1. Summary statistics on anchoring susceptibility distribution
2. Correlation analysis between personality traits and susceptibility
3. Profiles of extreme cases (most/least susceptible)
4. Initial hypotheses for Week 8 testing

**Coming up next:** Social Proof and Herding Behavior (Week 8)

---

---
# Section 3: Exercise 2 - Social Proof and Herding Behavior
**Week 8 | Duration: 60 minutes**

### What is Social Proof?
Social proof is the phenomenon where people conform to the actions of others, assuming those actions are correct. This can lead to herding behavior where people follow the crowd even against their private information.

### Research Questions:
1. Which participants are most influenced by social proof?
2. Does "Need for Uniqueness" predict resistance to social proof?
3. How does agreeableness relate to conformity?
4. What marketing strategies leverage social proof effectively?

In [ ]:
# CELL 3.1: Create Social Proof Susceptibility Score
# Social proof susceptibility relates to:
# - High agreeableness (going along with others)
# - High neuroticism (uncertainty, looking to others for guidance)
# - Low openness (less independent thinking)

print("Creating social proof susceptibility score...\n")

# Normalize personality traits
df['agreeableness_norm'] = (df['agreeableness'] - df['agreeableness'].min()) / (df['agreeableness'].max() - df['agreeableness'].min())
df['neuroticism_norm'] = (df['neuroticism'] - df['neuroticism'].min()) / (df['neuroticism'].max() - df['neuroticism'].min())
df['openness_norm'] = (df['openness'] - df['openness'].min()) / (df['openness'].max() - df['openness'].min())

# Create social proof susceptibility
# High agreeableness + High neuroticism + Low openness = High susceptibility
df['social_proof_susceptibility'] = (
    (0.4 * df['agreeableness_norm']) +
    (0.3 * df['neuroticism_norm']) +
    (0.3 * (1 - df['openness_norm']))  # Invert openness
)

# Create categories
df['social_proof_category'] = pd.cut(df['social_proof_susceptibility'],
                                      bins=3,
                                      labels=['Independent', 'Moderate', 'Conformist'])

print("✓ Social proof susceptibility score created!\n")
print("Distribution of Social Proof Categories:")
print(df['social_proof_category'].value_counts().sort_index())
print(f"\nSummary Statistics:")
print(df['social_proof_susceptibility'].describe().round(3))

In [ ]:
# CELL 3.2: Visualize Social Proof Susceptibility

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Distribution
axes[0, 0].hist(df['social_proof_susceptibility'].dropna(), bins=30, edgecolor='black', alpha=0.7, color='purple')
axes[0, 0].set_xlabel('Social Proof Susceptibility Score')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Social Proof Susceptibility')
axes[0, 0].axvline(df['social_proof_susceptibility'].median(), color='red', linestyle='--', linewidth=2, label='Median')
axes[0, 0].legend()

# Plot 2: vs Agreeableness
axes[0, 1].scatter(df['agreeableness'], df['social_proof_susceptibility'], alpha=0.3, color='purple')
axes[0, 1].set_xlabel('Agreeableness')
axes[0, 1].set_ylabel('Social Proof Susceptibility')
axes[0, 1].set_title('Agreeableness vs Social Proof Susceptibility')
corr = df[['agreeableness', 'social_proof_susceptibility']].corr().iloc[0, 1]
axes[0, 1].text(0.05, 0.95, f'r = {corr:.3f}', transform=axes[0, 1].transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 3: vs Neuroticism
axes[1, 0].scatter(df['neuroticism'], df['social_proof_susceptibility'], alpha=0.3, color='orange')
axes[1, 0].set_xlabel('Neuroticism')
axes[1, 0].set_ylabel('Social Proof Susceptibility')
axes[1, 0].set_title('Neuroticism vs Social Proof Susceptibility')
corr = df[['neuroticism', 'social_proof_susceptibility']].corr().iloc[0, 1]
axes[1, 0].text(0.05, 0.95, f'r = {corr:.3f}', transform=axes[1, 0].transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 4: Category distribution
category_counts = df['social_proof_category'].value_counts().sort_index()
colors_sp = ['green', 'orange', 'red']
axes[1, 1].bar(range(len(category_counts)), category_counts.values, color=colors_sp, alpha=0.7)
axes[1, 1].set_xticks(range(len(category_counts)))
axes[1, 1].set_xticklabels(category_counts.index, rotation=45, ha='right')
axes[1, 1].set_ylabel('Number of Participants')
axes[1, 1].set_title('Participants by Social Proof Category')

for i, v in enumerate(category_counts.values):
    axes[1, 1].text(i, v + 10, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 🔴 COLLABORATIVE EXERCISE 3.1: Find Conformists vs. Independents

**Task:** Identify participants who are most vs. least influenced by social proof.

**Discussion:** How would you market differently to these two groups?

In [ ]:
# CELL 3.3: Identify Extreme Cases for Social Proof

# Most conformist
most_conformist = df.nlargest(5, 'social_proof_susceptibility')[['pid', 'social_proof_susceptibility',
                                                                   'agreeableness', 'neuroticism', 'openness',
                                                                   'extraversion']]

# Most independent
most_independent = df.nsmallest(5, 'social_proof_susceptibility')[['pid', 'social_proof_susceptibility',
                                                                     'agreeableness', 'neuroticism', 'openness',
                                                                     'extraversion']]

print("="*70)
print("MOST CONFORMIST (High Social Proof Susceptibility)")
print("="*70)
print(most_conformist.to_string(index=False))
print()

print("="*70)
print("MOST INDEPENDENT (Low Social Proof Susceptibility)")
print("="*70)
print(most_independent.to_string(index=False))

# Profile the most conformist participant
most_conformist_pid = most_conformist.iloc[0]['pid']
conformist_profile = df[df['pid'] == most_conformist_pid].iloc[0]

print("\n" + "="*70)
print(f"DETAILED PROFILE: MOST CONFORMIST (PID {most_conformist_pid})")
print("="*70)
print(f"Social Proof Susceptibility: {conformist_profile['social_proof_susceptibility']:.3f}")
print(f"Agreeableness: {conformist_profile['agreeableness']:.3f}")
print(f"Neuroticism: {conformist_profile['neuroticism']:.3f}")
print(f"Openness: {conformist_profile['openness']:.3f}")
print(f"Extraversion: {conformist_profile['extraversion']:.3f}")
print(f"Conscientiousness: {conformist_profile['conscientiousness']:.3f}")

### 🟡 DISCUSSION POINT 4: Marketing with Social Proof

**Instructor: Lead 15-minute discussion**

**Real-world examples:**
- "10,000+ customers love this product"
- "Best seller" badges
- Customer review counts and star ratings
- "X people are viewing this right now"
- "Y people bought this in the last 24 hours"

**Questions:**
1. How would you market to the "Conformist" segment?
2. How would you market to the "Independent" segment?
3. When does social proof backfire?
4. Are there products where independence is more valued than conformity?

In [ ]:
# CELL 3.4: Cross-Bias Analysis - Anchoring AND Social Proof
# Let's find participants with interesting combinations

print("="*70)
print("CROSS-BIAS ANALYSIS: Anchoring × Social Proof")
print("="*70)
print()

# Create 2x2 matrix of susceptibility types
df['bias_profile'] = 'Unknown'

high_anchor_thresh = df['anchoring_susceptibility'].quantile(0.67)
low_anchor_thresh = df['anchoring_susceptibility'].quantile(0.33)
high_social_thresh = df['social_proof_susceptibility'].quantile(0.67)
low_social_thresh = df['social_proof_susceptibility'].quantile(0.33)

# Define profiles
df.loc[(df['anchoring_susceptibility'] >= high_anchor_thresh) &
       (df['social_proof_susceptibility'] >= high_social_thresh), 'bias_profile'] = 'Highly Susceptible (Both)'

df.loc[(df['anchoring_susceptibility'] <= low_anchor_thresh) &
       (df['social_proof_susceptibility'] <= low_social_thresh), 'bias_profile'] = 'Highly Resistant (Both)'

df.loc[(df['anchoring_susceptibility'] >= high_anchor_thresh) &
       (df['social_proof_susceptibility'] <= low_social_thresh), 'bias_profile'] = 'Anchor-Prone Independent'

df.loc[(df['anchoring_susceptibility'] <= low_anchor_thresh) &
       (df['social_proof_susceptibility'] >= high_social_thresh), 'bias_profile'] = 'Social-Prone Rational'

print("Distribution of Bias Profiles:")
print(df['bias_profile'].value_counts())
print()

# Find examples of each interesting profile
for profile in ['Highly Susceptible (Both)', 'Highly Resistant (Both)',
                'Anchor-Prone Independent', 'Social-Prone Rational']:
    profile_df = df[df['bias_profile'] == profile]
    if len(profile_df) > 0:
        example = profile_df.iloc[0]
        print(f"\n{profile}:")
        print(f"  Example PID: {example['pid']}")
        print(f"  Anchoring: {example['anchoring_susceptibility']:.3f}")
        print(f"  Social Proof: {example['social_proof_susceptibility']:.3f}")
        print(f"  Present Bias: {example['present_bias']:.3f}")
        print(f"  Agreeableness: {example['agreeableness']:.3f}")

### ⏸️ PAUSE - End of Social Proof Exercise

**Progress Check:**
- You've now analyzed TWO behavioral biases
- You've identified participant segments for each
- You're ready for the third bias: Framing Effects

---

---
# Section 4: Exercise 3 - Framing Effects
**Week 8 | Duration: 50 minutes**

### What is Framing?
Framing effects occur when people react differently to the same information depending on how it's presented. Gain framing ("save $50") versus loss framing ("don't lose $50") can dramatically change decisions.

### Research Questions:
1. Who is most susceptible to framing effects?
2. Does loss aversion predict frame susceptibility?
3. How can we leverage framing in marketing messages?
4. When should we use gain vs. loss frames?

In [ ]:
# CELL 4.1: Create Framing Susceptibility Score
# Framing susceptibility relates to:
# - High loss aversion (sensitive to losses)
# - High neuroticism (emotional reactivity)
# - Low conscientiousness (less analytical processing)

print("Creating framing susceptibility score...\n")

# Normalize relevant traits
df['loss_aversion_norm'] = (df['loss_aversion'] - df['loss_aversion'].min()) / (df['loss_aversion'].max() - df['loss_aversion'].min())

# Create framing susceptibility
df['framing_susceptibility'] = (
    (0.5 * df['loss_aversion_norm']) +
    (0.3 * df['neuroticism_norm']) +
    (0.2 * (1 - df['conscientiousness_norm']))
)

# Create categories
df['framing_category'] = pd.cut(df['framing_susceptibility'],
                                 bins=3,
                                 labels=['Frame-Resistant', 'Moderately Influenced', 'Frame-Susceptible'])

print("✓ Framing susceptibility score created!\n")
print("Distribution of Framing Categories:")
print(df['framing_category'].value_counts().sort_index())
print(f"\nSummary Statistics:")
print(df['framing_susceptibility'].describe().round(3))

In [ ]:
# CELL 4.2: Visualize Framing Susceptibility

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Distribution
axes[0, 0].hist(df['framing_susceptibility'].dropna(), bins=30, edgecolor='black', alpha=0.7, color='teal')
axes[0, 0].set_xlabel('Framing Susceptibility Score')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Framing Susceptibility')
axes[0, 0].axvline(df['framing_susceptibility'].median(), color='red', linestyle='--', linewidth=2, label='Median')
axes[0, 0].legend()

# Plot 2: vs Loss Aversion
axes[0, 1].scatter(df['loss_aversion'], df['framing_susceptibility'], alpha=0.3, color='teal')
axes[0, 1].set_xlabel('Loss Aversion')
axes[0, 1].set_ylabel('Framing Susceptibility')
axes[0, 1].set_title('Loss Aversion vs Framing Susceptibility')
corr = df[['loss_aversion', 'framing_susceptibility']].corr().iloc[0, 1]
axes[0, 1].text(0.05, 0.95, f'r = {corr:.3f}', transform=axes[0, 1].transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 3: vs Neuroticism
axes[1, 0].scatter(df['neuroticism'], df['framing_susceptibility'], alpha=0.3, color='darkgreen')
axes[1, 0].set_xlabel('Neuroticism')
axes[1, 0].set_ylabel('Framing Susceptibility')
axes[1, 0].set_title('Neuroticism vs Framing Susceptibility')
corr = df[['neuroticism', 'framing_susceptibility']].corr().iloc[0, 1]
axes[1, 0].text(0.05, 0.95, f'r = {corr:.3f}', transform=axes[1, 0].transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Plot 4: Category distribution
category_counts = df['framing_category'].value_counts().sort_index()
colors_fr = ['green', 'yellow', 'red']
axes[1, 1].bar(range(len(category_counts)), category_counts.values, color=colors_fr, alpha=0.7)
axes[1, 1].set_xticks(range(len(category_counts)))
axes[1, 1].set_xticklabels(category_counts.index, rotation=45, ha='right')
axes[1, 1].set_ylabel('Number of Participants')
axes[1, 1].set_title('Participants by Framing Category')

for i, v in enumerate(category_counts.values):
    axes[1, 1].text(i, v + 10, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 🔴 COLLABORATIVE EXERCISE 4.1: Gain vs. Loss Frame Recommendations

**Task:** For each framing susceptibility category, determine when to use gain vs. loss frames.

**Consider:**
- Loss aversion levels
- Neuroticism
- Risk aversion
- Product category implications

In [ ]:
# CELL 4.3: Identify Frame-Susceptible Participants

# Most frame-susceptible
most_frame_suscept = df.nlargest(5, 'framing_susceptibility')[['pid', 'framing_susceptibility',
                                                                 'loss_aversion', 'loss_aversion_pct',
                                                                 'neuroticism', 'risk_aversion']]

# Most frame-resistant
most_frame_resistant = df.nsmallest(5, 'framing_susceptibility')[['pid', 'framing_susceptibility',
                                                                    'loss_aversion', 'loss_aversion_pct',
                                                                    'neuroticism', 'risk_aversion']]

print("="*70)
print("MOST FRAME-SUSCEPTIBLE")
print("="*70)
print(most_frame_suscept.to_string(index=False))
print()

print("="*70)
print("MOST FRAME-RESISTANT")
print("="*70)
print(most_frame_resistant.to_string(index=False))

# Profile most susceptible
most_suscept_pid = most_frame_suscept.iloc[0]['pid']
suscept_profile = df[df['pid'] == most_suscept_pid].iloc[0]

print("\n" + "="*70)
print(f"DETAILED PROFILE: MOST FRAME-SUSCEPTIBLE (PID {most_suscept_pid})")
print("="*70)
print(f"Framing Susceptibility: {suscept_profile['framing_susceptibility']:.3f}")
print(f"Loss Aversion: {suscept_profile['loss_aversion']:.3f} ({suscept_profile['loss_aversion_pct']}th percentile)")
print(f"Neuroticism: {suscept_profile['neuroticism']:.3f}")
print(f"Risk Aversion: {suscept_profile['risk_aversion']:.3f}")
print(f"Present Bias: {suscept_profile['present_bias']:.3f}")
print("\nMARKETING RECOMMENDATION:")
print("→ Use LOSS FRAMING: 'Don't miss out', 'Limited time', 'Before it's gone'")
print("→ Emphasize what they stand to LOSE by not acting")
print("→ Create urgency and scarcity messaging")

### 🟡 DISCUSSION POINT 5: Framing in Real Marketing

**Instructor: Lead 15-minute discussion**

**Real examples to analyze:**

**Gain Frames:**
- "Save 30% on your order"
- "Earn 2% cashback"
- "Get a free gift with purchase"

**Loss Frames:**
- "Don't lose your discount - expires today"
- "Last chance - only 3 left"
- "Without insurance, you risk losing everything"

**Questions:**
1. Which products work better with loss frames?
2. Which customer segments respond to each frame?
3. Can you over-use loss framing?
4. Ethical considerations?

In [ ]:
# CELL 4.4: Compare All Three Biases
# Let's see which participants are susceptible to MULTIPLE biases

print("="*70)
print("TRI-BIAS ANALYSIS: Anchoring × Social Proof × Framing")
print("="*70)
print()

# Find participants in top tercile for ALL THREE biases
triple_suscept = df[
    (df['anchoring_susceptibility'] >= df['anchoring_susceptibility'].quantile(0.67)) &
    (df['social_proof_susceptibility'] >= df['social_proof_susceptibility'].quantile(0.67)) &
    (df['framing_susceptibility'] >= df['framing_susceptibility'].quantile(0.67))
]

print(f"Participants susceptible to ALL THREE biases: {len(triple_suscept)} ({len(triple_suscept)/len(df)*100:.1f}%)")
print()

if len(triple_suscept) > 0:
    print("Top 5 'Triple-Susceptible' Participants:")
    print("-"*70)

    triple_top = triple_suscept.nlargest(5, 'anchoring_susceptibility')[[
        'pid', 'anchoring_susceptibility', 'social_proof_susceptibility', 'framing_susceptibility',
        'present_bias_pct', 'loss_aversion_pct', 'agreeableness', 'neuroticism'
    ]]

    print(triple_top.to_string(index=False))

    print("\n" + "="*70)
    print("MARKETING INSIGHT: 'High-Value Persuadable' Segment")
    print("="*70)
    print(f"Segment Size: {len(triple_suscept):,} participants ({len(triple_suscept)/len(df)*100:.1f}% of sample)")
    print("\nCharacteristics:")
    print(f"  - Average Present Bias: {triple_suscept['present_bias'].mean():.3f} (vs. {df['present_bias'].mean():.3f} overall)")
    print(f"  - Average Loss Aversion: {triple_suscept['loss_aversion'].mean():.3f} (vs. {df['loss_aversion'].mean():.3f} overall)")
    print(f"  - Average Agreeableness: {triple_suscept['agreeableness'].mean():.3f} (vs. {df['agreeableness'].mean():.3f} overall)")
    print(f"  - Average Neuroticism: {triple_suscept['neuroticism'].mean():.3f} (vs. {df['neuroticism'].mean():.3f} overall)")
    print("\nMarketing Strategy:")
    print("  1. Use high anchor prices before showing sale price")
    print("  2. Show social proof (reviews, bestseller badges, customer counts)")
    print("  3. Use loss framing ('Limited time', 'Don't miss out')")
    print("  4. Combine all three for maximum persuasion")

# Find participants resistant to ALL THREE biases
triple_resistant = df[
    (df['anchoring_susceptibility'] <= df['anchoring_susceptibility'].quantile(0.33)) &
    (df['social_proof_susceptibility'] <= df['social_proof_susceptibility'].quantile(0.33)) &
    (df['framing_susceptibility'] <= df['framing_susceptibility'].quantile(0.33))
]

print("\n" + "="*70)
print(f"\nParticipants resistant to ALL THREE biases: {len(triple_resistant)} ({len(triple_resistant)/len(df)*100:.1f}%)")
print("="*70)
print("\nMARKETING INSIGHT: 'Rational Decision-Maker' Segment")
print("\nStrategy:")
print("  1. Focus on objective product features and specifications")
print("  2. Provide detailed comparison charts")
print("  3. Emphasize value proposition and ROI")
print("  4. Avoid manipulative tactics - they'll see through them")

### ⏸️ PAUSE - End of Week 8 Exercises

**Week 8 Deliverable Preview:**
For your hypothesis testing document, include:
1. Statistical tests for all three biases
2. Correlation matrices between biases and personality traits
3. Segment profiles for each bias type
4. Cross-bias analysis (participants susceptible to multiple biases)
5. Visualizations showing key relationships

**Coming up next:** Individual Marketing Applications (Week 9)

---

---
# Section 5: Individual Marketing Application Development
**Week 9 | Duration: Individual work + sharing**

### Your Task:
Each student must develop ONE unique marketing application based on the behavioral economics insights from the Twin-2K-500 dataset.

### Requirements:
1. **Must be data-driven:** Reference specific participant IDs and their data
2. **Must be unique:** Find a pattern or segment no one else has identified
3. **Must be specific:** Include concrete marketing tactics, not just general ideas
4. **Must be actionable:** Something a real company could implement

### This section provides tools and examples to help you develop your application.

### 🔴 EXERCISE 5.1: Find Your Unique Segment

Use the cells below to explore the dataset and find interesting patterns.

**Ideas for unique segments:**
1. Participants with CONFLICTING traits (e.g., high loss aversion BUT low framing susceptibility)
2. Age × bias interactions
3. Multiple bias combinations that create unique opportunity
4. Participants who defy theoretical predictions
5. Micro-segments with specific personality+bias combinations

In [ ]:
# CELL 5.1: Exploration Template - Find Conflicting Traits
# Example: High loss aversion but LOW framing susceptibility (counterintuitive!)

# Find participants with high loss aversion but low framing susceptibility
conflicted = df[
    (df['loss_aversion'] > df['loss_aversion'].quantile(0.75)) &
    (df['framing_susceptibility'] < df['framing_susceptibility'].quantile(0.25))
]

print(f"Found {len(conflicted)} 'Conflicted' participants")
print("High loss aversion BUT resistant to framing\n")

if len(conflicted) > 0:
    print("Sample participants:")
    print(conflicted[['pid', 'loss_aversion', 'loss_aversion_pct',
                      'framing_susceptibility', 'conscientiousness', 'neuroticism']].head(10))

    print("\n" + "="*70)
    print("MARKETING APPLICATION IDEA:")
    print("="*70)
    print("This segment is loss-averse but analytical enough to resist framing tricks.")
    print("\nStrategy:")
    print("  - Acknowledge their loss aversion: 'We know protection is important to you'")
    print("  - Provide objective risk data and statistics")
    print("  - Don't use manipulative language - be transparent")
    print("  - Position as 'smart protection' not 'fear-based selling'")
    print("\nProduct fit: Insurance, security products, backup services")
else:
    print("Try adjusting the quantile thresholds to find this segment")

In [ ]:
# CELL 5.2: Custom Exploration Cell
# Use this cell to explore YOUR unique segment
# Modify the conditions below to find interesting patterns

# EXAMPLE: Find participants who are...
# - High on anchoring
# - Low on social proof
# - High present bias

my_segment = df[
    (df['anchoring_susceptibility'] > df['anchoring_susceptibility'].quantile(0.70)) &
    (df['social_proof_susceptibility'] < df['social_proof_susceptibility'].quantile(0.30)) &
    (df['present_bias'] > df['present_bias'].quantile(0.70))
]

print(f"My Segment Size: {len(my_segment)} participants\n")

if len(my_segment) > 0:
    print("Segment Characteristics:")
    print("-"*70)
    print(f"Average Anchoring Susceptibility: {my_segment['anchoring_susceptibility'].mean():.3f}")
    print(f"Average Social Proof Susceptibility: {my_segment['social_proof_susceptibility'].mean():.3f}")
    print(f"Average Present Bias: {my_segment['present_bias'].mean():.3f} ({my_segment['present_bias_pct'].mean():.0f}th percentile)")
    print(f"Average Loss Aversion: {my_segment['loss_aversion'].mean():.3f}")
    print(f"\nPersonality Profile:")
    print(f"  Openness: {my_segment['openness'].mean():.3f}")
    print(f"  Conscientiousness: {my_segment['conscientiousness'].mean():.3f}")
    print(f"  Extraversion: {my_segment['extraversion'].mean():.3f}")
    print(f"  Agreeableness: {my_segment['agreeableness'].mean():.3f}")
    print(f"  Neuroticism: {my_segment['neuroticism'].mean():.3f}")

    print("\n" + "="*70)
    print("SAMPLE PARTICIPANT IDs FOR YOUR APPLICATION:")
    print("="*70)
    print(my_segment['pid'].head(5).tolist())

    print("\n💡 Use these participant IDs in your marketing application!")
    print("   Show their specific data to justify your strategy.")
else:
    print("No participants match these criteria. Try adjusting the thresholds.")

# TODO for your individual project:
# 1. Modify the conditions above to find YOUR unique segment
# 2. Analyze what makes this segment special
# 3. Develop specific marketing tactics
# 4. Reference specific participant IDs as examples

In [ ]:
# CELL 5.3: Detailed Participant Profile for Case Study
# Select ONE participant to feature in your marketing application

# Replace this with a PID from your segment
featured_pid = my_segment['pid'].iloc[0] if len(my_segment) > 0 else df['pid'].iloc[0]

featured_profile = df[df['pid'] == featured_pid].iloc[0]

print("="*70)
print(f"DETAILED PROFILE: PARTICIPANT {featured_pid}")
print("="*70)
print()
print("BEHAVIORAL ECONOMICS SCORES:")
print(f"  Anchoring Susceptibility: {featured_profile['anchoring_susceptibility']:.3f}")
print(f"  Social Proof Susceptibility: {featured_profile['social_proof_susceptibility']:.3f}")
print(f"  Framing Susceptibility: {featured_profile['framing_susceptibility']:.3f}")
print()
print("UNDERLYING TRAITS:")
print(f"  Present Bias: {featured_profile['present_bias']:.3f} ({featured_profile['present_bias_pct']}th percentile)")
print(f"  Loss Aversion: {featured_profile['loss_aversion']:.3f} ({featured_profile['loss_aversion_pct']}th percentile)")
print(f"  Risk Aversion: {featured_profile['risk_aversion']:.3f} ({featured_profile['risk_aversion_pct']}th percentile)")
print()
print("BIG FIVE PERSONALITY:")
print(f"  Openness: {featured_profile['openness']:.3f}")
print(f"  Conscientiousness: {featured_profile['conscientiousness']:.3f}")
print(f"  Extraversion: {featured_profile['extraversion']:.3f}")
print(f"  Agreeableness: {featured_profile['agreeableness']:.3f}")
print(f"  Neuroticism: {featured_profile['neuroticism']:.3f}")
print()
print("="*70)
print("Use this profile in your marketing application write-up!")
print("="*70)

### 🟡 DISCUSSION POINT 6: Sharing Individual Applications

**Instructor: Facilitate sharing session (60 minutes)**

Each student presents (10 minutes):
1. **The segment they identified** (with participant IDs)
2. **Why this segment is interesting/unique**
3. **Specific marketing tactics** for this segment
4. **Expected outcomes** based on behavioral insights
5. **Q&A from group**

**Evaluation criteria:**
- ✓ Uses specific participant data
- ✓ Identifies unique pattern
- ✓ Provides concrete marketing tactics
- ✓ Connects to behavioral economics theory
- ✓ Actionable for real companies

---
# Section 6: Final Analysis and Export
**Week 9 | Prepare final deliverables**

This section helps you prepare your final report by generating summary statistics and visualizations.

In [ ]:
# CELL 6.1: Generate Correlation Matrix for All Biases

# Select key variables
key_vars = ['anchoring_susceptibility', 'social_proof_susceptibility', 'framing_susceptibility',
            'present_bias', 'loss_aversion', 'risk_aversion',
            'openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism']

# Create correlation matrix
corr_matrix = df[key_vars].corr()

# Visualize
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix: Behavioral Economics Biases and Personality Traits', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Correlation matrix saved as 'correlation_matrix.png'")
print("\nInclude this in your final report!")

In [ ]:
# CELL 6.2: Generate Summary Table for Report

summary_data = {
    'Bias Type': ['Anchoring', 'Social Proof', 'Framing'],
    'Mean Score': [
        df['anchoring_susceptibility'].mean(),
        df['social_proof_susceptibility'].mean(),
        df['framing_susceptibility'].mean()
    ],
    'Std Dev': [
        df['anchoring_susceptibility'].std(),
        df['social_proof_susceptibility'].std(),
        df['framing_susceptibility'].std()
    ],
    'High Susceptibility (N)': [
        (df['anchoring_category'] == 'High Susceptibility').sum(),
        (df['social_proof_category'] == 'Conformist').sum(),
        (df['framing_category'] == 'Frame-Susceptible').sum()
    ],
    'High Susceptibility (%)': [
        (df['anchoring_category'] == 'High Susceptibility').sum() / len(df) * 100,
        (df['social_proof_category'] == 'Conformist').sum() / len(df) * 100,
        (df['framing_category'] == 'Frame-Susceptible').sum() / len(df) * 100
    ]
}

summary_df = pd.DataFrame(summary_data)
print("="*70)
print("SUMMARY TABLE FOR FINAL REPORT")
print("="*70)
print(summary_df.to_string(index=False))
print()

# Save to CSV
summary_df.to_csv('bias_summary_table.csv', index=False)
print("✓ Summary table saved as 'bias_summary_table.csv'")

In [ ]:
# CELL 6.3: Export Your Segment Data
# Save your custom segment for reference

# If you found a segment in Cell 5.2, save it here
if len(my_segment) > 0:
    # Select relevant columns
    export_cols = ['pid', 'anchoring_susceptibility', 'social_proof_susceptibility', 'framing_susceptibility',
                   'present_bias', 'present_bias_pct', 'loss_aversion', 'loss_aversion_pct',
                   'risk_aversion', 'risk_aversion_pct',
                   'openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism']

    my_segment_export = my_segment[export_cols]
    my_segment_export.to_csv('my_segment_data.csv', index=False)

    print(f"✓ Exported {len(my_segment_export)} participants from your segment")
    print("  Saved as: my_segment_data.csv")
    print("\nUse this data to:")
    print("  1. Reference specific participant IDs in your write-up")
    print("  2. Create additional visualizations")
    print("  3. Support your marketing application with data")
else:
    print("No segment to export. Run Cell 5.2 first to identify your segment.")

---
## 🎉 Congratulations!

You've completed the behavioral economics digital twin exercises!

### What You've Accomplished:
1. ✓ Loaded and explored the Twin-2K-500 dataset (2,058 participants)
2. ✓ Analyzed three behavioral economics biases:
   - Anchoring and Adjustment
   - Social Proof and Herding
   - Framing Effects
3. ✓ Identified individual differences in bias susceptibility
4. ✓ Connected personality traits to behavioral biases
5. ✓ Developed unique marketing applications
6. ✓ Generated data-driven insights for real-world applications

### Next Steps for Your Final Report:
1. **Introduction (1 page):** Brief overview of digital twins and behavioral economics
2. **Behavioral Economics Analysis (4-5 pages):**
   - Anchoring findings
   - Social proof findings
   - Framing findings
   - Cross-bias analysis
3. **Individual Marketing Applications (4 pages):** Each student contributes 1 page
4. **Synthesis and Conclusions (1-2 pages):** Key takeaways and implications
5. **References**

### Files Generated:
- `correlation_matrix.png` - For your report
- `bias_summary_table.csv` - For your report
- `my_segment_data.csv` - For your individual application

**Good luck with your final report!** 🚀

---